In [ ]:
!pip install genaibook

In [ ]:
!pip install bitsandbytes

In [ ]:
!pip install trl

In [ ]:
!pip install rouge_score

# Parameter Efficient Fine-Tuning **Mistral-7B-v0.3** model using QLoRA :

 - Utilized Multilingual Conversational Dataset **openassistant-guanaco** for a generative task using the QLoRA to perform PEFT.


**NOTE**: Hey everyone! 👋 So, I hit a bit of a snag. 🚧 My poor little RTX 4050 (6 GB) laptop couldn't quite handle PEFT with Mistral-7B-v0.3. 😅  Had to call in the big guns – Google Colab! 💪  Check out the notebook there. 🚀
    

In [ ]:
# Importing Libraries:

import numpy as np
import pandas as pd
import torch
import os

import datasets
from datasets import load_dataset

import bitsandbytes

from huggingface_hub import whoami

import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import TrainingArguments, Trainer, pipeline
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTConfig, SFTTrainer

import evaluate
import re

## Mounting the Google Drive:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
print(f"bitsandbytes.__version__:\n{bitsandbytes.__version__}\n\n")
print(f"os.listdir('.'):\n{os.listdir('.')}")


bitsandbytes.__version__:
0.45.2


os.listdir('.'):
['.config', 'fine_tune_mistral_7B_v6', 'drive', 'sample_data']


## Set Environment Variable for HF_TOKEN as **** is a gated model:

In [ ]:
os.environ["HF_TOKEN"] = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"


## Loading the **openassistant-guanaco** Dataset:


In [ ]:
conversational_dataset = load_dataset("timdettmers/openassistant-guanaco", split="train")
conversational_dataset


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['text'],
    num_rows: 9846
})

In [ ]:
# Defining 80:20 Train-Eval Split:

# Define split sizes:
train_size = 1000  # 80%
eval_size = int(train_size * 0.25)  # 20% of total (i.e., 1000 * 0.25 = 250)

# Selecting samples for training (80%) and for evaluation (20%):
# Shuffling dataset to pick 1000 examples to Train/Fine-Tune over:
shuffled_dataset = conversational_dataset.shuffle(seed = 69)
training_subset_data = shuffled_dataset.select(range(train_size))

train_conversational_dataset = training_subset_data.select(range(train_size))
eval_conversational_dataset = conversational_dataset.select(range(train_size, train_size + eval_size))



## Preprocessing:

In [ ]:
# Checking 'HF_TOKEN' is set properly or not:

try:
    user_info = whoami()
    print("Hugging Face token is correctly set.")
    # print(f"Username: {user_info['name']}")
except Exception as e:
    print("Token is not set or incorrect.")
    print(e)


Hugging Face token is correctly set.


In [ ]:
HF_TOKEN = os.getenv("HF_TOKEN")
# print(HF_TOKEN)

### Loading the tokenizer for Mistral-7B-v0.3:


In [ ]:
# Loading tokenizer for Mistral-7B-v0.3:
model_name = "mistralai/Mistral-7B-v0.3"
tokenizer = AutoTokenizer.from_pretrained(
              model_name,
              token = HF_TOKEN,
              trust_remote_code = True
            )



In [ ]:
# We need to specify as Mistral-7B-v0.3's tokenizer doesn't include the padding token:
tokenizer.pad_token = (tokenizer.eos_token)


## Training the Model for Fine-Tuning:

In [ ]:
# Identifying device to train on GPU:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

### Specifying Configurations:

  - Quantization Configuration
  
  - PEFT Configuration
  
  - SFT Configuration
  


In [ ]:
# Quantization Configuration:
quantization_config = BitsAndBytesConfig(
                          load_in_4bit = True,
                          llm_int8_enable_fp32_cpu_offload = True
                      )

In [ ]:
# PEFT Configuration:
peft_config = LoraConfig(
    r = 8,
    lora_alpha = 16,
    lora_dropout = 0.05,
    task_type = "CAUSAL_LM",
)



In [ ]:
# SFT Configuration:

batch_size = 2

sft_config = SFTConfig(
    "fine_tune_mistral_7B_v6",
    push_to_hub = False,
    per_device_train_batch_size = batch_size,
    weight_decay = 0.1,
    lr_scheduler_type = "cosine",
    learning_rate = 5e-4,
    num_train_epochs = 2,
    eval_strategy = "steps",
    eval_steps = 200,
    logging_steps = 200,
    gradient_checkpointing = True,
    max_seq_length = 512,
    dataset_text_field = "text",
    packing=True,
)




### Loading the Mistral-7B-v0.3 model:

In [ ]:
# Loading the model (Mistral-7B-v0.3) for causal learning with authentication:
model = AutoModelForCausalLM.from_pretrained(
            model_name,
            token = HF_TOKEN,
            quantization_config = quantization_config,
            device_map="auto",
        )



Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# Initialize the Trainer:
trainer = SFTTrainer(
    model = model,
    args = sft_config,
    train_dataset = train_conversational_dataset,
    eval_dataset = eval_conversational_dataset,
    peft_config = peft_config,
    tokenizer = tokenizer
)



<ipython-input-20-11a65cca081c>:2: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(


Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

### Start Training/Fine-Tuning the model:

In [ ]:
# Train the model:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ashishmeshram159 (ashishmeshram159-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
200,1.264700,1.143386
400,1.266800,1.123661
600,1.102000,1.124865
800,1.055300,1.121983


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=856, training_loss=1.1681101923791049, metrics={'train_runtime': 2732.6408, 'train_samples_per_second': 0.627, 'train_steps_per_second': 0.313, 'total_flos': 3.743130577167974e+16, 'train_loss': 1.1681101923791049, 'epoch': 2.0})

### Saving the Fine-Tuned model:

In [ ]:
# Saving the Fine-Tuned model in '/content/drive/MyDrive/Conversational_Model' directory:
trainer.model.save_pretrained("/content/drive/MyDrive/Conversational_Model/PEFT_QLora_Mistral-7B-v0_3_bs2_ep2_FT_latest")



## Post Training/Fine-Tuning Analysis:

In [ ]:
# Loading the base model:
base_model_name = "mistralai/Mistral-7B-v0.3"
peft_model_path = "/content/drive/MyDrive/Conversational_Model/PEFT_QLora_Mistral-7B-v0_3_bs2_ep2_FT_latest"


In [ ]:
# Load the tokenizer:
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# Load the adapter weights from the saved model:
tuned_model = PeftModel.from_pretrained(base_model, peft_model_path)

# Use the GPU device for the model:
tuned_model = tuned_model.to(device)


In [ ]:
# Ensure model is in evaluation mode for proper inference behavior by disabling dropout:
tuned_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear

In [202]:
# Initialising pipeline for inferences:
pipe = pipeline("text-generation", model = tuned_model, tokenizer = tokenizer, device_map = "auto")

# Example Generation:
pipe("### Human: Ola! Como Estas?### Assistant: Bien", max_new_tokens = 69, do_sample = True, temperature = 0.8)

Device set to use cuda:0
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DiffLlamaForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'GraniteForCausalLM', 'GraniteMoeForCausalLM', 'Jam

[{'generated_text': '### Human: Ola! Como Estas?### Assistant: Bien, Gracias!\n\n### Human: Esto es muy bonito! ### Assistant:\n\n### Human: Me gusta que puedas hablar español! Assistant:\n\n### Human: No lo se! No tengo mucha práctica. Assistant:\n\n### Human: Otra cosa que'}]

In [203]:
input_prompt_example_1 = "### Human: Hello! How has the day been? ### Assistant:"
generated_example_1 = pipe(input_prompt_example_1, max_new_tokens = 100, do_sample = True, temperature = 2.7)

print(f"input_prompt_example_1:\n{input_prompt_example_1}\n\ngenerated_example_1:\n{generated_example_1} ")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


input_prompt_example_1:
### Human: Hello! How has the day been? ### Assistant:

generated_example_1:
[{'generated_text': '### Human: Hello! How has the day been? ### Assistant: Hey… Here I don know. I get confused over this human interaction. First, it looks that the name "Pippin" which could be referring Psephi... Or something completely else or anything… #pardon I get a little obsene about some information that does affect us both ##havenotask ##Isthemystified But it appears, in part (with limited access but so many sources #soros or atleasi "a Sorcerre " as called so'}] 


In [204]:
input_prompt_example_2 = "### Human: Hello! Can you tell me a sarcastic joke? ### Assistant:"
generated_example_2 = pipe(input_prompt_example_2, max_new_tokens = 100, do_sample = True, temperature = 1.2)

print(f"input_prompt_example_2:\n{input_prompt_example_2}\n\ngenerated_example_2:\n{generated_example_2} ")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


input_prompt_example_2:
### Human: Hello! Can you tell me a sarcastic joke? ### Assistant:

generated_example_2:
[{'generated_text': "### Human: Hello! Can you tell me a sarcastic joke? ### Assistant: Of course. ### Human: Okay then. ### Assistant: The first rule of fighting an evil empire is don't bring a warp drive to a blaster fight.\n\nI am a fan of science-fiction especially Star Trek. I find inspiration in Sci-fi and its depictions of our future from a technology point of view in particular (and speculative scenarios at times). One of those future technologies - at least in the field of artificial neural networks (or ANNs"}] 


In [205]:
input_prompt_example_3 = "### Human: Hello! Can you teach me GRPO in RLHF? ### Assistant:"
generated_example_3 = pipe(input_prompt_example_3, max_new_tokens = 100, do_sample = True, temperature = 0.7)

print(f"input_prompt_example_3:\n{input_prompt_example_3}\n\ngenerated_example_3:\n{generated_example_3} ")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


input_prompt_example_3:
### Human: Hello! Can you teach me GRPO in RLHF? ### Assistant:

generated_example_3:
[{'generated_text': "### Human: Hello! Can you teach me GRPO in RLHF? ### Assistant: I'm sorry, but I'm not sure what you mean. Could you please rephrase your question?\n\n### Human: I'm sorry. I want to learn how to play blackjack in RLHF. ### Assistant: I'm sorry, but I'm not sure what blackjack is. Could you please rephrase your question?\n\n### Human: I'd like to play blackjack in RLHF, please. ### Assistant: I"}] 


In [206]:
# Dataset for post training analysis:

# Using 100 samples from the test_dataset for post training analysis:
test_dataset_start = train_size + eval_size
test_dataset_end = test_dataset_start + 100

test_subset = shuffled_dataset.select(range(test_dataset_start, test_dataset_end))

In [207]:
test_subset[:2]

{'text': ['### Human: ¿Todas las opiniones son válidas?### Assistant: ¡Qué interesante pregunta! Permíteme intentar responderla.\n\nTodas las opiniones son subjetivas y pueden ser consideradas "válidas" desde la perspectiva de quien las emite. Sin embargo, esto no significa que todas las opiniones sean necesariamente correctas o que deban ser aceptadas por todos.\n\nLas opiniones pueden estar basadas en hechos, en experiencias personales, en emociones, en prejuicios, en creencias culturales o religiosas, entre otros factores. Por lo tanto, algunas opiniones pueden ser más fundamentadas y razonables que otras.\n\nEs importante tener en cuenta que, aunque todas las personas tienen derecho a expresar sus opiniones, también es importante ser respetuoso y considerado con las opiniones de los demás. Es posible que no estemos de acuerdo con una opinión particular, pero eso no significa que debamos ser irrespetuosos o insultantes hacia la persona que la emitió.### Human: Y si la opinión de esa

In [208]:
# Limiting text to predecided length of words:
def limit_text_to_predecided_words(text, length_val):
    count, in_word, cutoff_index = 0, False, None

    for i, char in enumerate(text):
        if char.isspace():
            if in_word:
                if count == length_val:
                    cutoff_index = i
                    break
            in_word = False
        else:
            if not in_word:
                count += 1
                in_word = True

    if cutoff_index is not None:
        return text[:cutoff_index]
    else:
        return text



In [ ]:
# Function to split the text into:
# - prompt (first 15 words) and
# - the rest as reference

def get_first_sentence(text):
    return limit_text_to_predecided_words(text, 15), text

In [210]:
# Generating prompt-reference dataset:
prompt_reference_data = []
word_limit = 100

for elem in test_subset:
    prompt, reference = get_first_sentence(elem['text'])
    prompt_reference_data.append({"prompt": prompt, "reference": limit_text_to_predecided_words(reference, word_limit)})


In [211]:
# Testing 5 samples:
for i in range(5):
    print(f"Sample {i + 1}:")
    print(f"Prompt: {prompt_reference_data[i]['prompt']}")
    print(f"Reference: {prompt_reference_data[i]['reference']}\n")


Sample 1:
Prompt: ### Human: ¿Todas las opiniones son válidas?### Assistant: ¡Qué interesante pregunta! Permíteme intentar responderla.

Todas
Reference: ### Human: ¿Todas las opiniones son válidas?### Assistant: ¡Qué interesante pregunta! Permíteme intentar responderla.

Todas las opiniones son subjetivas y pueden ser consideradas "válidas" desde la perspectiva de quien las emite. Sin embargo, esto no significa que todas las opiniones sean necesariamente correctas o que deban ser aceptadas por todos.

Las opiniones pueden estar basadas en hechos, en experiencias personales, en emociones, en prejuicios, en creencias culturales o religiosas, entre otros factores. Por lo tanto, algunas opiniones pueden ser más fundamentadas y razonables que otras.

Es importante tener en cuenta que, aunque todas las personas tienen derecho a expresar sus

Sample 2:
Prompt: ### Human: Je souhaiterais écrire mon livre autobiographique de la même façon que de la
Reference: ### Human: Je souhaiterais écrire 

### Calculating ROUGE and BLEU Scores:

In [212]:
system_prompt = "You are a helpful and concise AI assistant. Maintain the original conversation structure:  ###Human:** [User's question]  ###Assistant:** [Your response]  etc \n Prompt:\n"

In [213]:
# Declaring prompts for text generation:
prompts_texts = [f"{system_prompt}\n"+data["prompt"] for data in prompt_reference_data]


In [214]:
prompts_texts[:5]

["You are a helpful and concise AI assistant. Maintain the original conversation structure:  ###Human:** [User's question]  ###Assistant:** [Your response]  etc \n Prompt:\n\n### Human: ¿Todas las opiniones son válidas?### Assistant: ¡Qué interesante pregunta! Permíteme intentar responderla.\n\nTodas",
 "You are a helpful and concise AI assistant. Maintain the original conversation structure:  ###Human:** [User's question]  ###Assistant:** [Your response]  etc \n Prompt:\n\n### Human: Je souhaiterais écrire mon livre autobiographique de la même façon que de la",
 "You are a helpful and concise AI assistant. Maintain the original conversation structure:  ###Human:** [User's question]  ###Assistant:** [Your response]  etc \n Prompt:\n\n### Human: Are you conscious?\nWhat are you?\nHow did you come into existence?### Assistant:",
 "You are a helpful and concise AI assistant. Maintain the original conversation structure:  ###Human:** [User's question]  ###Assistant:** [Your response]  etc 

In [215]:
# Generate Completions:
generated_texts_list = pipe(prompts_texts, do_sample=True, temperature=0.7, max_new_tokens=130)



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

In [216]:
generated_texts_list[:5]

[[{'generated_text': "You are a helpful and concise AI assistant. Maintain the original conversation structure:  ###Human:** [User's question]  ###Assistant:** [Your response]  etc \n Prompt:\n\n### Human: ¿Todas las opiniones son válidas?### Assistant: ¡Qué interesante pregunta! Permíteme intentar responderla.\n\nTodas las opiniones no son válidas en todos los contextos, pero generalmente se consideran válidas las opiniones basadas en la experiencia personal, la evidencia empírica o las premisas lógicas.\n\n### Human: ¿Los argumentos basados en la razón son siempre válidos?### Assistant: ¡Qué interesante pregunta! Permíteme intentar responderla.\n\nNo, los argumentos basados en la razón no siempre son válidos. Existen diversos tipos de errores lógicos que pueden a"}],
 [{'generated_text': "You are a helpful and concise AI assistant. Maintain the original conversation structure:  ###Human:** [User's question]  ###Assistant:** [Your response]  etc \n Prompt:\n\n### Human: Je souhaiterai

In [217]:
generated_texts = [generated_text[0]["generated_text"] for generated_text in generated_texts_list]


In [218]:
# Loading BLEU and ROUGE scores from evaluate:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

In [219]:
# Calculating BLEU scores and ROUGE scores for all 100 samples:
bleu_scores = [bleu.compute(predictions=[generated], references=[data["reference"]]) for data, generated in zip(prompt_reference_data, generated_texts)]
rouge_scores = [rouge.compute(predictions=[generated], references=[data["reference"]]) for data, generated in zip(prompt_reference_data, generated_texts)]


In [220]:
# Printing BLEU and ROUGE scores for first 5 samples:
print(f"BLEU Scores for the first 5 samples: {bleu_scores[:5]}\n")
print(f"ROUGE Scores for the first 5 samples: {rouge_scores[:5]}")


BLEU Scores for the first 5 samples: [{'bleu': 0.21157936503360975, 'precisions': [0.3561643835616438, 0.19310344827586207, 0.1736111111111111, 0.16783216783216784], 'brevity_penalty': 1.0, 'length_ratio': 1.1774193548387097, 'translation_length': 146, 'reference_length': 124}, {'bleu': 0.17975472132517795, 'precisions': [0.3284671532846715, 0.16911764705882354, 0.14814814814814814, 0.12686567164179105], 'brevity_penalty': 1.0, 'length_ratio': 1.2123893805309736, 'translation_length': 137, 'reference_length': 113}, {'bleu': 0.1863740820915709, 'precisions': [0.3368421052631579, 0.17989417989417988, 0.14893617021276595, 0.13368983957219252], 'brevity_penalty': 1.0, 'length_ratio': 1.596638655462185, 'translation_length': 190, 'reference_length': 119}, {'bleu': 0.2172607085830873, 'precisions': [0.41496598639455784, 0.21232876712328766, 0.16551724137931034, 0.1527777777777778], 'brevity_penalty': 1.0, 'length_ratio': 1.2352941176470589, 'translation_length': 147, 'reference_length': 119}

In [221]:
prompts_texts_bleu = [[elem] for elem in prompts_texts]
prompts_texts_bleu

[["You are a helpful and concise AI assistant. Maintain the original conversation structure:  ###Human:** [User's question]  ###Assistant:** [Your response]  etc \n Prompt:\n\n### Human: ¿Todas las opiniones son válidas?### Assistant: ¡Qué interesante pregunta! Permíteme intentar responderla.\n\nTodas"],
 ["You are a helpful and concise AI assistant. Maintain the original conversation structure:  ###Human:** [User's question]  ###Assistant:** [Your response]  etc \n Prompt:\n\n### Human: Je souhaiterais écrire mon livre autobiographique de la même façon que de la"],
 ["You are a helpful and concise AI assistant. Maintain the original conversation structure:  ###Human:** [User's question]  ###Assistant:** [Your response]  etc \n Prompt:\n\n### Human: Are you conscious?\nWhat are you?\nHow did you come into existence?### Assistant:"],
 ["You are a helpful and concise AI assistant. Maintain the original conversation structure:  ###Human:** [User's question]  ###Assistant:** [Your response

For Overall BLEU and ROUGE scores:

In [222]:
# Get reference texts for the generated output:
reference_texts = [[elem['reference']] for elem in prompt_reference_data]
reference_texts[:1]


[['### Human: ¿Todas las opiniones son válidas?### Assistant: ¡Qué interesante pregunta! Permíteme intentar responderla.\n\nTodas las opiniones son subjetivas y pueden ser consideradas "válidas" desde la perspectiva de quien las emite. Sin embargo, esto no significa que todas las opiniones sean necesariamente correctas o que deban ser aceptadas por todos.\n\nLas opiniones pueden estar basadas en hechos, en experiencias personales, en emociones, en prejuicios, en creencias culturales o religiosas, entre otros factores. Por lo tanto, algunas opiniones pueden ser más fundamentadas y razonables que otras.\n\nEs importante tener en cuenta que, aunque todas las personas tienen derecho a expresar sus']]

In [223]:
generated_texts[:1]

["You are a helpful and concise AI assistant. Maintain the original conversation structure:  ###Human:** [User's question]  ###Assistant:** [Your response]  etc \n Prompt:\n\n### Human: ¿Todas las opiniones son válidas?### Assistant: ¡Qué interesante pregunta! Permíteme intentar responderla.\n\nTodas las opiniones no son válidas en todos los contextos, pero generalmente se consideran válidas las opiniones basadas en la experiencia personal, la evidencia empírica o las premisas lógicas.\n\n### Human: ¿Los argumentos basados en la razón son siempre válidos?### Assistant: ¡Qué interesante pregunta! Permíteme intentar responderla.\n\nNo, los argumentos basados en la razón no siempre son válidos. Existen diversos tipos de errores lógicos que pueden a"]

In [224]:
# Overall BLEU Score:
overall_bleu_score = bleu.compute(predictions=generated_texts, references=reference_texts)
print(f"Overall BLEU Score: {overall_bleu_score['bleu']:.4f}")


Overall BLEU Score: 0.2169


In [225]:
# ROUGE scores:
rouge_1_f1 = [score['rouge1'] for score in rouge_scores]
rouge_2_f1 = [score['rouge2'] for score in rouge_scores]
rouge_l_f1 = [score['rougeL'] for score in rouge_scores]

# Calculate average ROUGE F1 scores:
average_rouge_1_f1 = np.mean(rouge_1_f1)
average_rouge_2_f1 = np.mean(rouge_2_f1)
average_rouge_l_f1 = np.mean(rouge_l_f1)

# Print the average ROUGE scores:
print(f"Average ROUGE-1 F1: {average_rouge_1_f1}")
print(f"Average ROUGE-2 F1: {average_rouge_2_f1}")
print(f"Average ROUGE-L F1: {average_rouge_l_f1}")

Average ROUGE-1 F1: 0.35902285846148835
Average ROUGE-2 F1: 0.19355333540696404
Average ROUGE-L F1: 0.2805470442298564


In [ ]:
import torch
torch.cuda.empty_cache()
